# 5장 · KL 발산 — 바가지 초과분의 이름 ⚖️

웹 챕터 **"KL 발산"** 과 짝을 이루는 Python 실습입니다.
웹의 비대칭 저울에서 잡은 감을, 여기서는 **두 정의의 일치 확인·비대칭 산점도·증류 미니 실험**으로 굳힙니다.

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch5')

## 1. KL 함수: 두 정의가 같은지부터 확인

KL 은 두 가지로 쓸 수 있었죠:
- **빼기 정의**: $D(p\|q) = H(p,q) - H(p)$ (요금 전체 − 기본요금)
- **한 줄 정의**: $D(p\|q) = \sum p \cdot \log_2 \frac{p}{q}$

둘이 정말 같은지 4장의 지도들로 검산합니다.

In [ ]:
import numpy as np

def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(np.sum(p * np.log2(1 / p)))

def cross_entropy(p, q):
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    if np.any(q[mask] <= 0):
        return float('inf')
    return float(np.sum(p[mask] * np.log2(1 / q[mask])))

def kl(p, q):
    """D(p‖q) = Σ p·log₂(p/q) — 한 줄 정의. p>0 인 칸에 q=0 이면 ∞."""
    p, q = np.asarray(p, dtype=float), np.asarray(q, dtype=float)
    mask = p > 0
    if np.any(q[mask] <= 0):
        return float('inf')
    return float(np.sum(p[mask] * np.log2(p[mask] / q[mask])))

p_world = np.array([1/2, 1/4, 1/8, 1/8])
maps = {
    '🎯 세상과 같은 지도': p_world,
    '⚖️ 균등 지도       ': np.array([1/4, 1/4, 1/4, 1/4]),
    '🙃 거꾸로 지도     ': np.array([1/8, 1/8, 1/4, 1/2]),
}

print('지도                | 빼기 정의 H(p,q)-H(p) | 한 줄 정의 Σp·log₂(p/q)')
print('-' * 68)
for name, q in maps.items():
    sub = cross_entropy(p_world, q) - entropy(p_world)
    one = kl(p_world, q)
    print(f'{name} | {sub:20.4f}비트 | {one:20.4f}비트')
    assert abs(sub - one) < 1e-9, '두 정의가 다르다?! (그럴 수 없음)'
print('\n✅ 두 정의가 완전히 일치합니다. 로그의 나눗셈 규칙(2부 7장) 덕분입니다.')

## 2. 비대칭: 방향을 바꾸면 값이 달라진다

무작위 분포 쌍 2,000개에 대해 D(p‖q) 와 D(q‖p) 를 재서 산점도로 뿌립니다.
대칭이라면 모든 점이 대각선 위에 있어야 하는데… 과연?

In [ ]:
import matplotlib.pyplot as plt

# 한글 폰트 설정 (안 하면 한글 제목/라벨이 네모로 깨짐)
# Windows: 'Malgun Gothic' / macOS: 'AppleGothic' / Linux: 'NanumGothic' 로 바꾸세요
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False  # 음수 부호(-) 깨짐 방지

rng = np.random.default_rng(42)

def random_dist(n=4):
    raw = rng.random(n) + 1e-3
    return raw / raw.sum()

pairs = [(random_dist(), random_dist()) for _ in range(2000)]
d_pq = np.array([kl(a, b) for a, b in pairs])
d_qp = np.array([kl(b, a) for a, b in pairs])

fig, ax = plt.subplots(figsize=(5.5, 5.5))
ax.scatter(d_pq, d_qp, s=6, alpha=0.35, color='#6366f1')
lim = max(d_pq.max(), d_qp.max()) * 1.05
ax.plot([0, lim], [0, lim], color='#ef4444', lw=1.5, linestyle='--', label='대칭이라면 이 선 위')
ax.set_xlabel('D(p||q) (비트)')
ax.set_ylabel('D(q||p) (비트)')
ax.set_title('무작위 분포 2,000쌍: 점들이 대각선을 벗어난다 = 비대칭')
ax.legend()
plt.show()

gap = np.abs(d_pq - d_qp)
print(f'두 방향이 (거의) 같은 쌍: {np.mean(gap < 0.01) * 100:.1f}%  (대부분은 다르다!)')
print(f'가장 크게 벌어진 쌍     : D(p‖q)={d_pq[gap.argmax()]:.3f} vs D(q‖p)={d_qp[gap.argmax()]:.3f}')

극단도 확인해 봅시다. 웹 챕터의 **∞ 대 1비트** 예제입니다.

In [ ]:
truth = np.array([0.5, 0.5])   # 진실: 반반 동전
belief = np.array([1.0, 0.0])  # 믿음: 항상 앞면

print(f'D(진실 ‖ 믿음) = {kl(truth, belief)}   ← 가능한 뒷면을 불가능 취급 → 무한대')
print(f'D(믿음 ‖ 진실) = {kl(belief, truth)}   ← 반대 방향은 멀쩡한 1비트')
assert np.isinf(kl(truth, belief)) and abs(kl(belief, truth) - 1) < 1e-9

## 3. "항상 0 이상, 0은 p=q 뿐": 무작위 검증

무작위 쌍 10,000개에서 KL 의 최소값을 찾아봅니다. 음수가 하나라도 나오면 큰일이고,
**가장 0에 가까웠던 쌍**은 서로 닮은 분포여야 합니다.

In [ ]:
pairs10k = [(random_dist(), random_dist()) for _ in range(10000)]
ds = np.array([kl(a, b) for a, b in pairs10k])

best = int(ds.argmin())
print(f'10,000쌍 중 최소 KL : {ds.min():.6f} 비트  (음수 없음!)')
print(f'그 쌍의 p           : {np.round(pairs10k[best][0], 3)}')
print(f'그 쌍의 q           : {np.round(pairs10k[best][1], 3)}')
print('→ KL 이 0에 가까우려면 두 분포가 닮아야 합니다.')
assert ds.min() >= -1e-12, 'KL 이 음수가 나왔다?! (그럴 수 없음)'

## 4. AI 미리보기 · 증류: 선생의 분포를 흉내내는 학생

선생 모델의 출력 분포 p 를 두 학생이 흉내냅니다.
학생 A 는 분포 전체를 배웠고, 학생 B 는 1등 답만 몰빵으로 외웠어요. 누가 선생과 가까울까요?

> 아래 출력에 나오는 RLHF 는 **R**einforcement **L**earning from **H**uman **F**eedback,
> **사람 피드백 기반 강화학습**의 약자입니다(6부에서 정식으로 배워요).

In [ ]:
tokens = ['고양이', '삵', '호랑이', '강아지', '자동차']
teacher   = np.array([0.60, 0.20, 0.10, 0.08, 0.02])   # 선생: 부드러운 확신 (닮은 동물에도 확률)
student_a = np.array([0.55, 0.22, 0.12, 0.08, 0.03])   # 학생 A: 분포 전체를 흉내
student_b = np.array([0.96, 0.01, 0.01, 0.01, 0.01])   # 학생 B: 1등 답만 몰빵

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), sharey=True)
for ax, dist, name in [(axes[0], teacher, '선생 p'),
                       (axes[1], student_a, f'학생 A: D(p||q)={kl(teacher, student_a):.3f}비트'),
                       (axes[2], student_b, f'학생 B: D(p||q)={kl(teacher, student_b):.3f}비트')]:
    ax.bar(tokens, dist, color='#6366f1', alpha=0.85)
    ax.set_title(name, fontsize=10)
    ax.tick_params(axis='x', labelsize=8)
axes[0].set_ylabel('확률')
fig.suptitle('증류: 부드러운 분포 전체를 배운 학생 A 가 선생과 훨씬 가깝다 (KL 작음)')
plt.tight_layout()
plt.show()

print(f'학생 A 의 KL: {kl(teacher, student_a):.4f} 비트  ← 고양이 vs 삵의 미묘한 차이까지 흉내')
print(f'학생 B 의 KL: {kl(teacher, student_b):.4f} 비트  ← 정답만 외운 딱딱한 학생')
print('\nRLHF 의 KL 페널티도 같은 자입니다. 튜닝 모델이 원본 분포에서 멀어진 정도를 재서 벌점.')

## 5. 직접 풀어보기 ✏️

4장 거꾸로 지도의 KL: 세상 p = (1/2, 1/4, 1/8, 1/8), 지도 q = (1/8, 1/8, 1/4, 1/2) 일 때
D(p‖q) 는 몇 비트일까요? (웹 챕터의 StepQuiz 와 같은 계산)

In [ ]:
# TODO: 아래 answer 를 여러분의 계산으로 바꾸고 실행하세요.
# 힌트: 요금 전체(4장에서 2.625비트) − 기본요금 H(p) — 또는 kl(...) 재사용

answer = kl([1/2, 1/4, 1/8, 1/8], [1/8, 1/8, 1/4, 1/2])
print('내 답:', answer)
assert abs(answer - 0.875) < 1e-9, '다시 계산해 보세요 — 2.625 − 1.75'
print('✅ 잘했어요! 최악의 지도가 문 바가지 0.875비트, 이게 KL 입니다.')

---
**오늘의 한 줄**: D(p‖q) = 요금 전체 − 기본요금 = 바가지 초과분.
항상 0 이상, 0은 p=q 뿐, 그리고 방향이 있다(비대칭).

다음 챕터에서 재료가 전부 합쳐집니다. **"AI 가 배운다" = 데이터 분포와 모델 분포의
KL 최소화**, 한 줄 정의의 완성. 🙂